In [1]:
import os
import re
import glob
import json
import subprocess
import numpy as np
import pandas as pd
import torch

In [2]:
INPUT = "/kaggle/input/datasets/nocturnalnerd18"
USERNAME = "ponishbhatia3"
TYPES = ["attribute", "relation"]
CONTROL_RATIO = 1.5
MIN_CONTROLS = 300
SHARES = {"true_yes": 0.6, "true_no": 0.4}
MAX_IMAGE_TOKENS = None
PLACEHOLDER_SEC_PER_QUESTION = 8.0


def batch_number(path):
    return int(re.search(r"_batch_(\d+)\.pt$", path).group(1))


with open(f"{INPUT}/vlm-questions/all_questions.json") as f:
    all_questions = json.load(f)
assert len(all_questions) == 17492, "attach vlm-questions version 2"
question_by_id = {q["question_id"]: q for q in all_questions}

with open(f"{INPUT}/vlm-labeled-examples/labeled_examples.json") as f:
    labels = pd.DataFrame(json.load(f))
assert "cache" in labels.columns, "attach vlm-labeled-examples version 3"
splits = labels[labels["cache"] == "question_aware"][["question_id", "split"]].drop_duplicates("question_id")

rows = []
for t in TYPES:
    paths = sorted(glob.glob(f"{INPUT}/dla-intern/{t}_batch_*.pt"), key=batch_number)
    assert paths, f"attach dla-intern ({t} batches missing)"
    for path in paths:
        for e in torch.load(path):
            rows.append({
                "question_id": e["question_id"], "type": e["hallucination_type"], "prefix": e["prefix"],
                "logit_diff": float(e["logit_diff"]), "in_pair": bool(e["in_pair"]), "n_image_tokens": int(e["n_image_tokens"]),
            })
rec = pd.DataFrame(rows)
assert rec.question_id.is_unique
rec["source"] = rec.prefix.str.rstrip("_")
n_all = len(rec)
rec = rec.merge(splits, on="question_id", how="inner")
assert len(rec) > 0.99 * n_all, "most dla-intern entries have no split"
print(n_all, "dla-intern entries,", len(rec), "joined to a split")

print(pd.crosstab([rec.type, rec.source], rec.in_pair, normalize="index").round(4))
rec = rec[rec.in_pair].copy()
rec["truth"] = rec.question_id.map(lambda q: question_by_id[q]["ground_truth_answer"].strip().lower())
rec["model_answer"] = np.where(rec.logit_diff > 0, "yes", "no")
rec["group"] = np.select(
    [
        (rec.truth == "yes") & (rec.model_answer == "yes"),
        (rec.truth == "no") & (rec.model_answer == "no"),
        (rec.truth == "no") & (rec.model_answer == "yes"),
        (rec.truth == "yes") & (rec.model_answer == "no"),
    ],
    ["true_yes", "true_no", "false_yes", "false_no"],
    default="other",
)
assert (rec.group != "other").all()
print(pd.crosstab(rec.n_image_tokens, [rec.type, rec.source]))
if MAX_IMAGE_TOKENS is not None:
    n_before = rec.groupby(["type", "source"]).size()
    fy_before = int((rec.group == "false_yes").sum())
    rec = rec[rec.n_image_tokens <= MAX_IMAGE_TOKENS].copy()
    print(f"image token cap {MAX_IMAGE_TOKENS}: kept per type and source", (rec.groupby(["type", "source"]).size() / n_before).round(3).to_dict())
    print("false_yes kept", int((rec.group == "false_yes").sum()), "of", fy_before)
print(len(rec), "questions with the top-1 token in the yes/no pair")
print(pd.crosstab([rec.type, rec.source, rec.split], rec.group))

7003 dla-intern entries, 7003 joined to a split
in_pair              False   True 
type      source                  
attribute amber     0.1058  0.8942
relation  amber     0.2066  0.7934
          reefknot  0.0000  1.0000
type           attribute relation         
source             amber    amber reefknot
n_image_tokens                            
256                  147       25      142
768                  114       14      149
1024                   2        0        5
1280                  48       12        3
1792                1363      257     1323
2304                  35        7        2
2560                  20        2        1
2816                   3        1       15
3328                1267      258     1283
6498 questions with the top-1 token in the yes/no pair
group                     false_yes  true_no  true_yes
type      source   split                              
attribute amber    test          72      159       236
                   train        284      

In [3]:
parts = []
for t in TYPES:
    d = rec[rec.type == t]
    fy = d[d.group == "false_yes"]
    parts.append(fy.assign(stratum="false_yes"))
    budget = max(int(CONTROL_RATIO * len(fy)), MIN_CONTROLS)
    for g, share in SHARES.items():
        pool = d[d.group == g]
        if len(pool) == 0:
            continue
        quota = min(int(budget * share), len(pool))
        parts.append(pool.groupby(["source", "split"], group_keys=False).sample(frac=quota / len(pool), random_state=0).assign(stratum=g))

selection = pd.concat(parts).sample(frac=1.0, random_state=0).reset_index(drop=True)
assert selection.question_id.is_unique
assert (selection.stratum == "false_yes").sum() == int((rec.group == "false_yes").sum()), "false_yes rows were dropped"

print(pd.crosstab([selection.type, selection.source, selection.split], selection.stratum, margins=True))
print("selected per type:", selection.groupby("type").size().to_dict(), "of", rec.groupby("type").size().to_dict())
print(f"{len(selection)} questions at a placeholder {PLACEHOLDER_SEC_PER_QUESTION} s/question: about "
      f"{len(selection) * PLACEHOLDER_SEC_PER_QUESTION / 3600:.1f} h of collection (replace with the measured speed)")

stratum                   false_yes  true_no  true_yes   All
type      source   split                                    
attribute amber    test          72       38        56   166
                   train        284      171       260   715
                   val           57       38        55   150
relation  amber    test           1       37        42    80
                   train          4      162       192   358
                   val            3       39        35    77
          reefknot test         188       86       136   410
                   train        905      378       631  1914
                   val          204       80       138   422
All                            1718     1029      1545  4292
selected per type: {'attribute': 1031, 'relation': 3261} of {'attribute': 2999, 'relation': 3499}
4292 questions at a placeholder 8.0 s/question: about 9.5 h of collection (replace with the measured speed)


In [4]:
SEL_DIR = "/kaggle/working/atp_intern_selection"
os.makedirs(SEL_DIR, exist_ok=True)

with open(os.path.join(SEL_DIR, "selection.json"), "w") as f:
    json.dump({
        "selected": {t: selection[selection.type == t].question_id.tolist() for t in TYPES},
        "stratum": dict(zip(selection.question_id, selection.stratum)),
        "group": dict(zip(selection.question_id, selection.group)),
        "source": dict(zip(selection.question_id, selection.source)),
        "split": dict(zip(selection.question_id, selection.split)),
        "prefix": dict(zip(selection.question_id, selection.prefix)),
        "n_image_tokens": dict(zip(selection.question_id, selection.n_image_tokens.astype(int).tolist())),
        "n_selected": int(len(selection)),
        "max_image_tokens": MAX_IMAGE_TOKENS,
    }, f)

with open(os.path.join(SEL_DIR, "dataset-metadata.json"), "w") as f:
    json.dump({
        "title": "atp-intern-selection",
        "id": f"{USERNAME}/atp-intern-selection",
        "subtitle": "Question subsample for InternVL3 attribution patching",
        "description": "Questions selected for InternVL3-8B attribution patching from the dla-intern answers (top-1 token in the yes/no pair): "
                       "every false_yes row (model said yes, truth no) plus stratified true_yes (60%) and true_no (40%) controls by source and split, "
                       "about 1.5 controls per false_yes row for each of attribute and relation. "
                       "stratum, group, source, split, prefix and n_image_tokens map each question_id to its selection group, label group, dataset, split and answer-token pair.",
        "licenses": [{"name": "CC0-1.0"}],
    }, f)


def _run(cmd):
    r = subprocess.run(cmd, capture_output=True, text=True)
    out = (r.stdout + r.stderr).strip()
    return r.returncode == 0 and "error" not in out.lower(), out


ok, out = _run(["kaggle", "datasets", "create", "-p", SEL_DIR])
if not ok:
    ok, out2 = _run(["kaggle", "datasets", "version", "-p", SEL_DIR, "-m", "selection"])
    out += "\n" + out2
print(out)
assert ok, "dataset upload failed"

Starting upload for file selection.json
Upload successful: selection.json (821KB)
Your private Dataset is being created. Please check progress at https://www.kaggle.com/datasets/ponishbhatia3/atp-intern-selection

  0%|          | 0.00/821k [00:00<?, ?B/s]
  2%|▏         | 16.0k/821k [00:00<00:08, 103kB/s]
100%|██████████| 821k/821k [00:02<00:00, 403kB/s]
